# Система прогнозирования пиковых нагрузок — полный пайплайн

**ВКР: «Разработка системы проактивного масштабирования веб-приложений»**

Ноутбук воспроизводит все эксперименты диплома по ячейкам:

| Секция | Глава ВКР | Что делает |
|--------|-----------|------------|
| §3.4 EDA | Глава 3 | Загрузка данных, разведочный анализ |
| §4.1 Модели | Глава 4 | XGBoost (Random Search) + LSTM (Random Search) + NeuralProphet (Grid Search) |
| §4.2 Пики | Глава 4 | Детекция пиковых нагрузок |
| §4.3 Walk-Forward | Глава 4 | Адаптивное переобучение с ADWIN-детектором дрейфа |

## 0. Установка пакетов и клонирование репозитория

In [ ]:
# Клонируем репозиторий (или обновляем если уже есть)
import os
if not os.path.exists('/content/Dyplom'):
    !git clone https://github.com/Roman-bib/Dyplom.git /content/Dyplom
else:
    !cd /content/Dyplom && git pull

%cd /content/Dyplom

# Установка зависимостей
!pip install -q xgboost neuralprophet tensorflow scikit-learn pandas numpy \
               matplotlib seaborn river joblib

## 1. Конфигурация — измени под свои данные

In [ ]:
import sys, os
sys.path.insert(0, '/content/Dyplom')

# ================================================================
# НАСТРОЙКА — меняй только эту секцию
# ================================================================

# Путь к CSV-файлу с данными
CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/SDV_hourly.csv"

# Директория для сохранения моделей, графиков, CSV
SAVE_DIR = "/content/results"

# FAST_MODE = True  →  пропустить NeuralProphet (экономит ~15 мин)
FAST_MODE = False

# Walk-Forward параметры
WF_N_FRESH    = 200   # принудительный retrain каждые N шагов
WF_CONFIRM_N  = 10    # подтверждение ADWIN (шагов подряд)
WF_TEST_LIMIT = 500   # сколько точек тестировать (None = все)

# ================================================================

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"SAVE_DIR: {SAVE_DIR}")
print(f"CSV_PATH: {CSV_PATH}")
print(f"FAST_MODE: {FAST_MODE}")

## 2. Монтирование Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## §3.4 — Загрузка данных и разведочный анализ (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from data_collection.csv_loader import load_csv

df = load_csv(CSV_PATH)

print(f"Загружено точек : {len(df)}")
print(f"Период          : {df['ds'].iloc[0]} → {df['ds'].iloc[-1]}")
print(f"min / max / mean: {df['y'].min():.0f} / {df['y'].max():.0f} / {df['y'].mean():.0f}")
print(f"NaN             : {df['y'].isna().sum()}")

# Сохраняем для следующих ячеек
df.to_csv(f'{SAVE_DIR}/raw_data.csv', index=False)

In [ ]:
# --- График временного ряда ---
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

ax = axes[0]
ax.plot(df['ds'], df['y'], linewidth=0.7, color='steelblue')
ax.set_title('Временной ряд нагрузки (полный)')
ax.set_ylabel('RPS')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax.grid(alpha=0.3)

ax = axes[1]
# Распределение значений
ax.hist(df['y'], bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(df['y'].mean(), color='red', linestyle='--', label=f"mean={df['y'].mean():.0f}")
ax.axvline(df['y'].quantile(0.95), color='orange', linestyle='--', label=f"p95={df['y'].quantile(0.95):.0f}")
ax.set_title('Распределение нагрузки')
ax.set_xlabel('RPS')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/eda_overview.png', dpi=150)
plt.show()
print(f"Сохранено: {SAVE_DIR}/eda_overview.png")

In [ ]:
# --- Суточный и недельный паттерн ---
df['hour']       = pd.to_datetime(df['ds']).dt.hour
df['dayofweek']  = pd.to_datetime(df['ds']).dt.dayofweek
dow_names = ['Пн','Вт','Ср','Чт','Пт','Сб','Вс']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
hourly = df.groupby('hour')['y'].mean()
ax.bar(hourly.index, hourly.values, color='steelblue', edgecolor='white')
ax.set_title('Средняя нагрузка по часам суток')
ax.set_xlabel('Час'); ax.set_ylabel('Среднее RPS')
ax.grid(alpha=0.3, axis='y')

ax = axes[1]
daily = df.groupby('dayofweek')['y'].mean()
ax.bar([dow_names[i] for i in daily.index], daily.values, color='orange', edgecolor='white')
ax.set_title('Средняя нагрузка по дням недели')
ax.set_ylabel('Среднее RPS')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/eda_seasonality.png', dpi=150)
plt.show()

# Убираем временные колонки
df.drop(columns=['hour', 'dayofweek'], inplace=True)

## Разбиение данных на train / val / test

In [ ]:
from preprocessing.feature_engineering import split_train_val_test

n = len(df)
TEST_H = min(480, n // 6)   # ~20%, но не более 480 часов
VAL_H  = TEST_H

train, val, test = split_train_val_test(df, test_hours=TEST_H, val_hours=VAL_H)
print(f"Train : {len(train)} точек  ({train['ds'].iloc[0]} → {train['ds'].iloc[-1]})")
print(f"Val   : {len(val)} точек  ({val['ds'].iloc[0]} → {val['ds'].iloc[-1]})")
print(f"Test  : {len(test)} точек  ({test['ds'].iloc[0]} → {test['ds'].iloc[-1]})")

# Сохраняем сплиты
train.to_csv(f'{SAVE_DIR}/train.csv', index=False)
val.to_csv(f'{SAVE_DIR}/val.csv', index=False)
test.to_csv(f'{SAVE_DIR}/test.csv', index=False)

# Визуализация сплита
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(train['ds'], train['y'], color='steelblue', linewidth=0.7, label=f'Train ({len(train)})')
ax.plot(val['ds'],   val['y'],   color='green',     linewidth=0.7, label=f'Val ({len(val)})')
ax.plot(test['ds'],  test['y'],  color='red',        linewidth=0.7, label=f'Test ({len(test)})')
ax.set_title('Разбиение на train / val / test')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/data_split.png', dpi=150)
plt.show()

## §4.1 — Сравнение моделей прогнозирования

Используется `models/comparison.py` (полная версия):
- **XGBoost** — Random Search по 7 гиперпараметрам (30 итераций)
- **LSTM** — Random Search по 5 гиперпараметрам (10 итераций), Huber loss
- **NeuralProphet** — Grid Search по 3 параметрам (8 комбинаций)

> `FAST_MODE = True` пропускает NeuralProphet (~15 мин)

In [ ]:
from models.comparison import ModelComparison
from preprocessing.feature_engineering import FeatureBuilder, _infer_step_minutes, _hours_to_periods

# Вычисляем window_size LSTM под частоту данных
step_min = _infer_step_minutes(pd.to_datetime(train['ds']))
lstm_window = _hours_to_periods(24, step_min)   # 24 часа в периодах
lstm_window = max(12, min(lstm_window, 288))
print(f"Шаг данных: {step_min} мин  |  LSTM window_size: {lstm_window}")

comparator = ModelComparison(model_save_dir=SAVE_DIR)
results = comparator.run(
    train, val, test,
    include_prophet  = not FAST_MODE,
    include_lstm     = True,
    lstm_window_size = lstm_window,
    force_retrain    = True,   # перепрогнать даже если файлы есть
)

comparator.save_best()
best_name, best_model = comparator.best_model()
print(f"\nЛучшая модель: {best_name}")

In [ ]:
# --- Сводная таблица метрик ---
import pandas as pd

SHOW_COLS = ['MAE', 'RMSE', 'MAPE', 'train_time_s']
rows = []
for name, m in comparator.results_.items():
    row = {'Модель': name}
    row.update({c: m.get(c, float('nan')) for c in SHOW_COLS})
    if 'mae_train' in m:
        row['MAE_train'] = m['mae_train']
        row['MAE_val']   = m['mae_val']
        row['overfit']   = m['overfit_ratio']
    rows.append(row)

metrics_df = pd.DataFrame(rows).set_index('Модель')
metrics_df = metrics_df.round(3)
metrics_df.to_csv(f'{SAVE_DIR}/metrics_comparison.csv')
print(f"Сохранено: {SAVE_DIR}/metrics_comparison.csv")
display(metrics_df)

In [ ]:
# --- График прогнозов всех моделей ---
from evaluation.metrics import plot_all_forecasts

plot_all_forecasts(
    train, val, test,
    predictions_dict = comparator.predictions_,
    save_path = f'{SAVE_DIR}/comparison_all_models.png',
    zoom = False,
)
plot_all_forecasts(
    train, val, test,
    predictions_dict = comparator.predictions_,
    save_path = f'{SAVE_DIR}/comparison_zoom.png',
    zoom = True,
)
plt.show()
print(f"Сохранено: {SAVE_DIR}/comparison_all_models.png")

### Важность признаков XGBoost

In [ ]:
import joblib
from models.xgboost_model import feature_importance
from preprocessing.feature_engineering import FeatureBuilder

xgb_model = comparator.models_.get('XGBoost') or joblib.load(f'{SAVE_DIR}/xgboost.pkl')
builder = FeatureBuilder()
X_train_fi, _ = builder.get_X_y(train)

imp_df = feature_importance(xgb_model, feature_names=list(X_train_fi.columns))
imp_df.to_csv(f'{SAVE_DIR}/feature_importance.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 6))
top12 = imp_df.head(12).iloc[::-1]   # перевернуть — лучшие наверху
bars = ax.barh(top12['feature'], top12['importance'], color='steelblue', edgecolor='white')
ax.set_title('Важность признаков XGBoost (gain), топ-12')
ax.set_xlabel('Importance (gain)')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/feature_importance.png', dpi=150)
plt.show()
print(imp_df.to_string(index=False))

### Доверительный интервал (квантильная регрессия 10–90%)

In [ ]:
from models.xgboost_model import get_confidence_interval
from evaluation.metrics import plot_forecast

builder_ci = FeatureBuilder()
(X_tr, y_tr), (X_vl, y_vl), (X_te, y_te) = builder_ci.transform_splits(train, val, test)

print("Обучение квантильных моделей (q=0.10 и q=0.90)...")
lower, upper = get_confidence_interval(
    X_tr, y_tr, X_vl, y_vl, X_te, save_dir=SAVE_DIR
)

xgb_preds = comparator.predictions_.get('XGBoost')
plot_forecast(
    train, val, test, xgb_preds, 'XGBoost',
    lower=lower, upper=upper,
    save_path=f'{SAVE_DIR}/forecast_xgb_ci.png',
    zoom=False,
)
plot_forecast(
    train, val, test, xgb_preds, 'XGBoost',
    lower=lower, upper=upper,
    save_path=f'{SAVE_DIR}/forecast_xgb_ci_zoom.png',
    zoom=True,
)
plt.show()

# Экспорт предсказаний с CI
pred_df = test[['ds', 'y']].iloc[:len(xgb_preds)].copy().reset_index(drop=True)
for name, preds in comparator.predictions_.items():
    pred_df[name] = np.array(preds[:len(pred_df)])
pred_df['XGBoost_lower'] = lower[:len(pred_df)]
pred_df['XGBoost_upper'] = upper[:len(pred_df)]
pred_df.to_csv(f'{SAVE_DIR}/predictions.csv', index=False)
print(f"Сохранено: {SAVE_DIR}/predictions.csv")

## §4.2 — Детекция пиковых нагрузок

In [ ]:
from evaluation.peak_detection import PeakDetector

rps_max = float(train['y'].max())
target_per_replica = rps_max / 10   # предполагаем MAX_REPLICAS = 10

detector = PeakDetector(
    method='rolling_std',
    k=2.0,
    target_rps_per_replica=target_per_replica,
    min_replicas=1,
    max_replicas=10,
)
detector.fit(train['y'])

builder_pk = FeatureBuilder()
X_test_pk, y_test_pk = builder_pk.get_X_y(test)
xgb_preds_pk = comparator.predictions_.get('XGBoost')
predicted_series = pd.Series(xgb_preds_pk[:len(y_test_pk)], index=y_test_pk.index)

events_df = detector.detect_series(y_test_pk, predicted_series)
summary_pk = detector.summary(events_df)

print(f"Метод: rolling_std  |  Порог: {summary_pk['threshold']:.0f} RPS")
print(f"Пиков: {summary_pk['peaks_detected']} / {summary_pk['total_points']} ({summary_pk['peak_ratio_pct']}%)")
print(f"Severity: {summary_pk['severity_counts']}")

events_df.to_csv(f'{SAVE_DIR}/peak_events.csv', index=False)
print(f"\nСохранено: {SAVE_DIR}/peak_events.csv")

In [ ]:
# --- График пиков ---
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(events_df['timestamp'], events_df['rps'], color='gray', linewidth=0.7, label='Факт')
ax.plot(events_df['timestamp'], events_df['predicted'], color='steelblue', linewidth=0.9, label='Прогноз XGBoost')
ax.axhline(summary_pk['threshold'], color='red', linestyle='--', linewidth=1.0, label=f"Порог {summary_pk['threshold']:.0f}")

peaks = events_df[events_df['is_peak']]
ax.scatter(peaks['timestamp'], peaks['rps'], color='red', s=12, zorder=5, label=f'Пики ({len(peaks)})')

ax.set_title('Детекция пиковых нагрузок на тестовом периоде')
ax.set_ylabel('RPS')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/peak_detection.png', dpi=150)
plt.show()

# Первые 10 пиков
if peaks.shape[0] > 0:
    print("\nПервые 10 пиков:")
    display(peaks[['timestamp','rps','predicted','severity','recommended_replicas']].head(10))

## §4.3 — Walk-Forward валидация с адаптивным переобучением

Система прогнозирует точку за точкой. При обнаружении концепт-дрейфа 
(ADWIN или принудительно каждые `N_FRESH` шагов) модель переобучается 
на накопленной истории.

Параллельно работает **фиксированная модель-baseline** (начальный XGBoost) 
для сравнения — это ключевой эксперимент главы 4.3.

In [ ]:
from evaluation.walk_forward import run_walk_forward
from retraining.scheduler import make_xgb_train_fn
from retraining.drift_detector import ADWINDriftDetector
from models.forecasters import predict_xgboost_wf
import joblib

# Начальная модель — лучший XGBoost после сравнения
initial_xgb = comparator.models_.get('XGBoost') or joblib.load(f'{SAVE_DIR}/xgboost.pkl')

builder_wf  = FeatureBuilder()
drift_det   = ADWINDriftDetector(n_fresh=WF_N_FRESH, confirmation_n=WF_CONFIRM_N)
train_fn_wf = make_xgb_train_fn(builder_wf, save_dir=SAVE_DIR)

test_wf = test.iloc[:WF_TEST_LIMIT].reset_index(drop=True) if WF_TEST_LIMIT else test.reset_index(drop=True)

print(f"Walk-forward: {len(test_wf)} шагов")
print(f"n_fresh={WF_N_FRESH}, confirmation_n={WF_CONFIRM_N}")
print()

wf_res = run_walk_forward(
    train          = train,
    val            = val,
    test           = test_wf,
    initial_model  = initial_xgb,
    predict_fn     = predict_xgboost_wf,
    train_fn       = train_fn_wf,
    builder        = builder_wf,
    drift_detector = drift_det,
    save_dir       = SAVE_DIR,
    verbose        = True,
)

In [ ]:
# --- Итоги walk-forward ---
res_df     = wf_res['results_df']
base_df    = wf_res['baseline_df']
summary_wf = wf_res['summary']
retrains   = summary_wf['retrain_timestamps']

print("=" * 50)
print("ИТОГ WALK-FORWARD")
print("=" * 50)
print(f"MAE адаптивная   : {summary_wf['mae_adaptive']}")
print(f"MAE фиксированная: {summary_wf['mae_baseline']}")
print(f"Улучшение        : {summary_wf['improvement_pct']}%")
print(f"Переобучений     : {summary_wf['n_retrains']}")
if summary_wf['retrain_intervals']:
    intervals = summary_wf['retrain_intervals']
    print(f"Интервалы (шаги) : min={min(intervals)}, max={max(intervals)}, mean={np.mean(intervals):.0f}")

In [ ]:
# --- Комбинированный график walk-forward ---
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# --- График 1: Факт vs Прогноз ---
ax1 = axes[0]
ax1.plot(res_df['timestamp'], res_df['y_true'],
         color='gray', linewidth=0.7, alpha=0.8, label='Факт')
ax1.plot(res_df['timestamp'], res_df['y_pred'],
         color='steelblue', linewidth=1.1, label='Адаптивная')
if 'y_pred_baseline' in base_df.columns:
    ax1.plot(base_df['timestamp'], base_df['y_pred_baseline'],
             color='orange', linewidth=1.0, linestyle='--', label='Фиксированная')
for i, ts in enumerate(retrains):
    ax1.axvline(ts, color='crimson', linestyle=':', linewidth=0.9, alpha=0.8,
                label='Переобучение' if i == 0 else None)
ax1.set_title('Walk-Forward: Прогноз vs Факт')
ax1.set_ylabel('RPS')
ax1.legend(loc='upper right'); ax1.grid(alpha=0.3)

# --- График 2: Скользящий MAE ---
ax2 = axes[1]
window_roll = min(24, len(res_df) // 10)
roll_adapt = res_df['mae'].rolling(window_roll, min_periods=1).mean()
roll_base  = base_df['mae_baseline'].rolling(window_roll, min_periods=1).mean()
ax2.plot(res_df['timestamp'], roll_adapt, color='steelblue', linewidth=1.3,
         label=f'MAE адаптивная (rolling {window_roll})')
ax2.plot(base_df['timestamp'], roll_base, color='orange', linewidth=1.2,
         linestyle='--', label=f'MAE фиксированная (rolling {window_roll})')
for ts in retrains:
    ax2.axvline(ts, color='crimson', linestyle=':', linewidth=0.9, alpha=0.7)
ax2.set_title('Скользящий MAE: адаптивная vs фиксированная модель')
ax2.set_ylabel('MAE'); ax2.set_xlabel('Время')
ax2.legend(loc='upper right'); ax2.grid(alpha=0.3)

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/walk_forward_combined.png', dpi=150)
plt.show()
print(f"Сохранено: {SAVE_DIR}/walk_forward_combined.png")

In [ ]:
# --- График MAE только (крупно, для диплома) ---
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(res_df['timestamp'], roll_adapt, color='steelblue', linewidth=1.5,
        label=f'Адаптивная модель (MAE={summary_wf["mae_adaptive"]})')
ax.plot(base_df['timestamp'], roll_base, color='orange', linewidth=1.4,
        linestyle='--', label=f'Фиксированная модель (MAE={summary_wf["mae_baseline"]})')
for i, ts in enumerate(retrains):
    ax.axvline(ts, color='crimson', linestyle=':', linewidth=1.0, alpha=0.7,
               label='Переобучение' if i == 0 else None)

improvement = summary_wf['improvement_pct']
ax.set_title(f'Walk-Forward MAE: адаптивная vs фиксированная  (улучшение {improvement}%, {len(retrains)} переобучений)')
ax.set_ylabel('MAE (скользящее среднее)')
ax.set_xlabel('Время')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.xticks(rotation=20)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/walk_forward_mae.png', dpi=150)
plt.show()

In [ ]:
# --- Лог переобучений ---
log_path = f'{SAVE_DIR}/walk_forward_log.csv'
if os.path.exists(log_path):
    log_df = pd.read_csv(log_path)
    print(f"События переобучения ({len(log_df)}):\n")
    display(log_df)
    print()

# Интервалы между переобучениями
if summary_wf['retrain_intervals']:
    intervals = summary_wf['retrain_intervals']
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(range(len(intervals)), intervals, color='steelblue', edgecolor='white')
    ax.axhline(np.mean(intervals), color='red', linestyle='--',
               label=f'Среднее: {np.mean(intervals):.0f} шагов')
    ax.set_title('Интервалы между переобучениями (шагов)')
    ax.set_xlabel('Номер переобучения')
    ax.set_ylabel('Шагов с предыдущего')
    ax.legend(); ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/retrain_intervals.png', dpi=150)
    plt.show()

## §4.4 — Итоговая сводка и экспорт

In [ ]:
print("=" * 60)
print("ИТОГОВАЯ СВОДКА ЭКСПЕРИМЕНТА")
print("=" * 60)

# --- §4.1: Сравнение моделей ---
print("\n§4.1  Сравнение моделей:")
display(metrics_df[['MAE','RMSE','MAPE']].round(2))

# --- §4.2: Пики ---
print(f"\n§4.2  Детекция пиков:")
print(f"  Порог: {summary_pk['threshold']:.0f} RPS")
print(f"  Пиков: {summary_pk['peaks_detected']} / {summary_pk['total_points']} ({summary_pk['peak_ratio_pct']}%)")
print(f"  Severity: {summary_pk['severity_counts']}")

# --- §4.3: Walk-forward ---
print(f"\n§4.3  Walk-Forward адаптивное переобучение:")
print(f"  MAE адаптивная   : {summary_wf['mae_adaptive']}")
print(f"  MAE фиксированная: {summary_wf['mae_baseline']}")
print(f"  Улучшение        : {summary_wf['improvement_pct']}%")
print(f"  Переобучений     : {summary_wf['n_retrains']}")

print(f"\nВсе файлы сохранены в: {SAVE_DIR}/")

# --- Итоговый CSV для Typst / таблицы диплома ---
summary_row = {
    'best_model':          best_name,
    **{f'{n}_mae': v.get('MAE') for n, v in comparator.results_.items()},
    'peak_threshold':      summary_pk['threshold'],
    'peaks_detected':      summary_pk['peaks_detected'],
    'wf_mae_adaptive':     summary_wf['mae_adaptive'],
    'wf_mae_baseline':     summary_wf['mae_baseline'],
    'wf_improvement_pct':  summary_wf['improvement_pct'],
    'wf_n_retrains':       summary_wf['n_retrains'],
}
pd.DataFrame([summary_row]).to_csv(f'{SAVE_DIR}/experiment_summary.csv', index=False)
print(f"Сохранено: {SAVE_DIR}/experiment_summary.csv")

In [ ]:
# --- Список всех сохранённых файлов ---
print(f"Файлы в {SAVE_DIR}/:")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:<45} {size/1024:.1f} KB")